# Tutorial 05 · CNNs in Practice: Features and Visual Explanations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_05_CNN_in_Practice/tutorial_05_cnn_practice.ipynb)

Reuse a pretrained MobileNetV2 for cats-versus-dogs classification, visualize filters and activations, and build Grad-CAM. Finish with an optional neural style-transfer experiment and a small vision-transformer shape demonstration.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** The PyTorch training loop and CNN concepts from Tutorials 03–04.

**Resources:** CPU supported in quick mode; optional GPU. Core downloads: cats/dogs data (~66 MB) and MobileNetV2 weights (~14 MB). Optional VGG19 style transfer downloads ~550 MB and uses more memory. No API key. Plan 60–90 minutes for reading and exercises.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals

Use the preprocessing associated with pretrained weights, freeze a feature extractor, train a new classifier, keep validation and test files separate, distinguish filters from activations, and interpret a class-specific heatmap cautiously.

The default core runs independently of earlier notebooks. `QUICK_RUN=True` uses 400 training / 100 validation / 100 test images. Larger mode uses 2,000 / 500 / 500. The style-transfer section is guarded by a separate flag and is skipped by default.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps an existing PyTorch/torchvision pair together and does not downgrade Colab’s installed stack. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'PIL': 'pillow'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import json, platform, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset, Subset
import torchvision
from torchvision import datasets, transforms, models
from PIL import Image
from IPython.display import display
%matplotlib inline

SEED = 42
QUICK_RUN = True
DEVICE_PREFERENCE = "auto"  # "cpu" forces CPU; "auto" uses CUDA when available.
assert DEVICE_PREFERENCE in {"auto", "cpu"}
device = torch.device("cuda" if DEVICE_PREFERENCE == "auto" and torch.cuda.is_available() else "cpu")
def seed_everything():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
seed_everything()
torch.set_num_threads(min(2, torch.get_num_threads()))
DATA_DIR = Path("data") / "tutorial_05"
OUTPUT_DIR = Path("outputs") / "tutorial_05"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
versions = {"python": platform.python_version(), "torch": str(torch.__version__),
            "torchvision": str(torchvision.__version__), "numpy": np.__version__,
            "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
            "device": str(device), "seed": SEED, "quick_run": QUICK_RUN}
display(pd.Series(versions, name="Environment"))
run_started = time.perf_counter()

RUN_STYLE_TRANSFER = False  # Optional: large VGG19 download; see Section 7.

## 2. Download images and split before building loaders

We use the filtered cats-and-dogs teaching archive hosted by Google. Its `train` folder supplies training images; its `validation` folder is divided into **disjoint validation and test subsets**. No loader is created before the split is finalized.

The archive is cached, checked for ZIP corruption, and extracted to the notebook’s data directory. Its SHA-256 is recorded with the results. This records the downloaded artifact; unlike a predeclared checksum, it does not independently authenticate its contents.

In [ ]:
import hashlib
import shutil
import urllib.request
import zipfile

DATA_URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
archive_path = DATA_DIR / "cats_and_dogs_filtered.zip"
if not archive_path.exists():
    partial = archive_path.with_suffix(".part")
    with urllib.request.urlopen(DATA_URL, timeout=60) as response, partial.open("wb") as destination:
        shutil.copyfileobj(response, destination)
    partial.replace(archive_path)
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
with zipfile.ZipFile(archive_path) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("Corrupted archive. Remove the cached ZIP and rerun.")
    root = DATA_DIR.resolve()
    for member in archive.infolist():
        if not (root / member.filename).resolve().is_relative_to(root):
            raise ValueError("Unsafe archive member path")
    archive.extractall(root)

dataset_root = DATA_DIR / "cats_and_dogs_filtered"
class_names = ["cats", "dogs"]
train_files, val_files, test_files = [], [], []
rng = np.random.default_rng(SEED)
for label, name in enumerate(class_names):
    training = sorted((dataset_root / "train" / name).glob("*.jpg"))
    held_out = sorted((dataset_root / "validation" / name).glob("*.jpg"))
    train_order = rng.permutation(len(training))
    held_order = rng.permutation(len(held_out))
    n_train = 200 if QUICK_RUN else 1000
    n_val = n_test = 50 if QUICK_RUN else 250
    assert len(training) >= n_train and len(held_out) >= n_val + n_test
    train_files.extend((training[i], label) for i in train_order[:n_train])
    val_files.extend((held_out[i], label) for i in held_order[:n_val])
    test_files.extend((held_out[i], label) for i in held_order[n_val:n_val+n_test])
train_paths, val_paths, test_paths = [{p.resolve() for p, _ in group} for group in [train_files,val_files,test_files]]
assert not train_paths & val_paths and not train_paths & test_paths and not val_paths & test_paths
print("Images per split:", len(train_files), len(val_files), len(test_files))
print("Archive SHA-256:", archive_sha256)

## 3. Match preprocessing to pretrained weights

MobileNetV2 was trained on ImageNet. Its weights enum supplies the corresponding resize, crop, pixel scaling, and channel normalization. We use that transform for every split. The pretrained 1,000-class head is not the cats/dogs classifier: we replace it with a new two-class head.

Only the new head is trained here. This is **feature extraction**, a form of transfer learning. Fine-tuning the backbone is a separate experiment. The original pretraining data may include similar animal categories; this is not training from scratch or evidence of generalization to every deployment setting.

In [ ]:
weights = models.MobileNet_V2_Weights.IMAGENET1K_V2
preprocess = weights.transforms()
pretrained = models.mobilenet_v2(weights=weights).to(device).eval()
backbone = pretrained.features
for parameter in backbone.parameters():
    parameter.requires_grad_(False)

class ImageFiles(Dataset):
    def __init__(self, files, transform):
        self.files, self.transform = files, transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, index):
        path, label = self.files[index]
        with Image.open(path) as image:
            return self.transform(image.convert("RGB")), label

image_train = ImageFiles(train_files, preprocess)
image_val = ImageFiles(val_files, preprocess)
image_test = ImageFiles(test_files, preprocess)
mean_rgb = torch.tensor(preprocess.mean).view(3, 1, 1)
std_rgb = torch.tensor(preprocess.std).view(3, 1, 1)
def rgb_image(normalized):
    return (normalized.detach().cpu() * std_rgb + mean_rgb).clamp(0, 1).permute(1, 2, 0).numpy()

print("Preprocessing:", preprocess)
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, index in zip(axes.flat, [0,1,2,len(image_train)//2,len(image_train)//2+1,len(image_train)//2+2]):
    image, label = image_train[index]
    ax.imshow(rgb_image(image)); ax.set_title(class_names[label]); ax.axis("off")
plt.tight_layout(); plt.show()

### Cache fixed features, then train a small head

The backbone produces spatial feature maps. Global average pooling averages each channel over height and width, yielding 1,280 features per image. We compute these fixed features once and train the two-class linear head on them. Batch normalization in the frozen backbone remains in evaluation mode.

The following standard training helper uses validation loss for checkpoint selection. Cached features are used only because the backbone and preprocessing stay fixed; changing either requires recomputing them.

In [ ]:
def make_loader(dataset, shuffle=False, batch_size=128):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

def one_epoch(model, loader, optimizer=None, augment=None):
    training = optimizer is not None
    model.train(training)
    loss_sum, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if training and augment is not None:
                images = augment(images)
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
            assert torch.isfinite(loss), "Non-finite loss"
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item()
            count += len(labels)
    return {"loss": loss_sum / count, "accuracy": correct / count}

def fit_classifier(model, training, validation, epochs, weight_decay=0.0, augment=None, lr=0.001):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    history, best_loss, best_state, best_epoch = [], float("inf"), None, 0
    for epoch in range(1, epochs + 1):
        train = one_epoch(model, training, optimizer, augment)
        val = one_epoch(model, validation)
        history.append({"epoch": epoch, "train_loss": train["loss"], "val_loss": val["loss"],
                        "train_accuracy": train["accuracy"], "val_accuracy": val["accuracy"]})
        if val["loss"] < best_loss:
            best_loss, best_epoch = val["loss"], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"Epoch {epoch}: training={train['accuracy']:.1%}, validation={val['accuracy']:.1%}")
    model.load_state_dict(best_state)
    model.eval()
    print(f"Restored validation-selected epoch {best_epoch}.")
    return pd.DataFrame(history), best_epoch

def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for prefix, label in [("train", "Training"), ("val", "Validation")]:
        axes[0].plot(history.epoch, history[prefix + "_loss"], label=label)
        axes[1].plot(history.epoch, history[prefix + "_accuracy"], label=label)
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1))
    for ax in axes: ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

In [ ]:
@torch.no_grad()
def extract_features(dataset):
    backbone.eval()
    values, targets = [], []
    for images, labels in make_loader(dataset, batch_size=32):
        maps = backbone(images.to(device))
        values.append(maps.mean(dim=(2, 3)).cpu())
        targets.append(labels)
    return TensorDataset(torch.cat(values), torch.cat(targets))

feature_train = extract_features(image_train)
feature_val = extract_features(image_val)
feature_test = extract_features(image_test)
assert feature_train.tensors[0].shape == (len(image_train), 1280)
print("Cached training features:", feature_train.tensors[0].shape)
seed_everything()
head = nn.Linear(1280, 2).to(device)
head_history, best_epoch = fit_classifier(head, make_loader(feature_train, shuffle=True, batch_size=32),
    make_loader(feature_val), epochs=20 if QUICK_RUN else 50, weight_decay=0.001, lr=0.003)
plot_history(head_history, "Frozen MobileNetV2 + trainable linear head")
validation_result = one_epoch(head, make_loader(feature_val))
test_result = one_epoch(head, make_loader(feature_test))
baseline_label = torch.bincount(feature_train.tensors[1]).argmax()
baseline_accuracy = (feature_test.tensors[1] == baseline_label).float().mean().item()
print("Validation:", validation_result, "Test:", test_result)
print("Training-majority baseline:", baseline_accuracy)

class FeatureClassifier(nn.Module):
    def __init__(self, features, classifier):
        super().__init__()
        self.features, self.classifier = features, classifier
    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=(2, 3)))

model = FeatureClassifier(backbone, head).to(device).eval()
with torch.no_grad():
    direct = model(image_test[0][0].unsqueeze(0).to(device)).cpu()
    cached = head(feature_test.tensors[0][:1].to(device)).cpu()
torch.testing.assert_close(direct, cached, rtol=1e-4, atol=1e-4)

## 4. Filters and activations answer different questions

A **filter** is a learned parameter tensor. An **activation** is the response to a particular input image. We show the first eight filters and channels, rather than selecting only visually interesting ones. Per-filter scaling helps display small weights but hides their relative magnitude.

In [ ]:
first_conv = backbone[0][0]
filters = first_conv.weight.detach().cpu()[:8]
fig, axes = plt.subplots(2, 4, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    filt = filters[i].permute(1, 2, 0)
    filt = (filt - filt.min()) / (filt.max() - filt.min()).clamp_min(1e-8)
    ax.imshow(filt); ax.set_title(f"Filter {i}"); ax.axis("off")
fig.suptitle("First-convolution RGB filters (individually scaled)")
plt.tight_layout(); plt.show()

example_image, example_label = image_test[0]
with torch.no_grad():
    early_maps = backbone[0](example_image.unsqueeze(0).to(device)).cpu()
fig, axes = plt.subplots(2, 4, figsize=(9, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(early_maps[0, i], cmap="viridis"); ax.set_title(f"Activation {i}"); ax.axis("off")
fig.suptitle(f"Early activations for a test {class_names[example_label]}")
plt.tight_layout(); plt.show()

## 5. Grad-CAM: where does a class score receive positive support?

For a chosen class logit $s_c$ and feature maps $A_k$, Grad-CAM averages $\partial s_c/\partial A_k$ over spatial positions to get channel weights, then computes $\operatorname{ReLU}(\sum_k\alpha_k A_k)$. We upsample the resulting coarse map onto the input crop.

We differentiate the **logit**, not a thresholded class label. Frozen parameters do not prevent gradients with respect to the input and intermediate activations. The forward hook is removed after each call to avoid accumulating state.

A heatmap is a model diagnostic, not a segmentation mask, causal explanation, or proof that the model reasoned correctly. We show maps for both classes to make the class dependence visible.

In [ ]:
def grad_cam(model, image, target_class):
    model.eval()
    captured = {}
    def capture(module, inputs, output):
        captured["maps"] = output
    hook = model.features[-1].register_forward_hook(capture)
    try:
        x = image.unsqueeze(0).to(device).detach().requires_grad_(True)
        logits = model(x)
        maps = captured["maps"]
        gradients = torch.autograd.grad(logits[0, target_class], maps)[0]
        channel_weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (channel_weights * maps).sum(dim=1, keepdim=True).relu()
        cam = F.interpolate(cam, size=image.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam[0, 0].detach().cpu()
        cam = cam / cam.max().clamp_min(1e-8)
        assert torch.isfinite(cam).all()
        return cam.numpy(), logits.detach().cpu().softmax(1)[0].numpy()
    finally:
        hook.remove()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
original = rgb_image(example_image)
axes[0].imshow(original); axes[0].set_title(f"True class: {class_names[example_label]}")
cam_summaries = {}
for target, ax in enumerate(axes[1:]):
    cam, probabilities = grad_cam(model, example_image, target)
    ax.imshow(original); ax.imshow(cam, cmap="magma", alpha=0.45, vmin=0, vmax=1)
    ax.set_title(f"{class_names[target]} score: {probabilities[target]:.1%}")
    cam_summaries[class_names[target]] = {"min":float(cam.min()), "max":float(cam.max()),
                                         "mean":float(cam.mean())}
for ax in axes: ax.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "gradcam.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. Save a reusable classifier

Save the backbone and head together with the weight identifier, transform settings, and class ordering. A CPU reload is checked against the original model’s logits. Reconstruct the architecture with `weights=None` before loading our saved state; this avoids another pretrained-weight download.

In [ ]:
checkpoint_path = OUTPUT_DIR / "cats_dogs_mobilenet.pt"
torch.save({"state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},
            "class_names":class_names, "pretrained_weights":"MobileNet_V2_Weights.IMAGENET1K_V2",
            "preprocessing":{"crop_size":preprocess.crop_size, "resize_size":preprocess.resize_size,
                             "mean":preprocess.mean, "std":preprocess.std}, "seed":SEED}, checkpoint_path)
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
restored = FeatureClassifier(models.mobilenet_v2(weights=None).features, nn.Linear(1280, 2)).eval()
restored.load_state_dict(saved["state_dict"])
with torch.no_grad():
    restored_logits = restored(example_image.unsqueeze(0))
    original_logits = model(example_image.unsqueeze(0).to(device)).cpu()
torch.testing.assert_close(restored_logits, original_logits, rtol=1e-4, atol=1e-4)
print("Checkpoint reload matches the original model.")

## 7. Optional: neural style transfer

Set `RUN_STYLE_TRANSFER=True` in the configuration cell and rerun from the top to enable this experiment. It downloads VGG19 weights (~550 MB); a GPU is helpful. The quick experiment uses 128 × 128 pixels and 40 optimization steps. It demonstrates the objective, not a polished artwork.

Here **image pixels** are optimized; pretrained network weights remain frozen. Content loss compares one feature map, and style loss compares Gram matrices (channel correlations) at several layers. Both targets are detached. Inputs stay in `[0,1]` and are normalized with ImageNet channel statistics before VGG; multiplying normalized inputs by 255 would be incorrect.

The content is the same dataset image used above. A procedurally generated color pattern supplies style, so no untracked local image files or third-party artwork downloads are needed.

In [ ]:
style_result = {"executed": False}
if RUN_STYLE_TRANSFER:
    size = 128 if QUICK_RUN else 256
    content = F.interpolate((example_image * std_rgb + mean_rgb).unsqueeze(0), size=(size,size),
                            mode="bilinear", align_corners=False).clamp(0,1).to(device)
    yy, xx = torch.meshgrid(torch.linspace(0,1,size), torch.linspace(0,1,size), indexing="ij")
    style = torch.stack([(torch.sin(30*xx)+1)/2, (torch.cos(24*yy)+1)/2,
                         (torch.sin(20*(xx+yy))+1)/2]).unsqueeze(0).to(device)
    vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features[:23].to(device).eval()
    for index, layer in enumerate(vgg):
        if isinstance(layer, nn.ReLU): vgg[index] = nn.ReLU(inplace=False)
    for parameter in vgg.parameters(): parameter.requires_grad_(False)
    normalization_mean = mean_rgb.unsqueeze(0).to(device)
    normalization_std = std_rgb.unsqueeze(0).to(device)
    style_layers, content_layer = ["0","5","10","19"], "21"
    def vgg_features(image):
        x = (image-normalization_mean)/normalization_std
        activations = {}
        for name, layer in vgg._modules.items():
            x = layer(x)
            if name in style_layers + [content_layer]: activations[name] = x
        return activations
    def gram(x):
        batch, channels, height, width = x.shape
        flattened = x.reshape(batch,channels,height*width)
        return flattened @ flattened.transpose(1,2) / (channels*height*width)
    with torch.no_grad():
        target_content = vgg_features(content)[content_layer].detach()
        target_style = {name:gram(value).detach() for name,value in vgg_features(style).items() if name in style_layers}
    generated = content.clone().requires_grad_(True)
    optimizer = torch.optim.Adam([generated], lr=0.03)
    losses = []
    for step in range(40 if QUICK_RUN else 150):
        optimizer.zero_grad(set_to_none=True)
        features = vgg_features(generated)
        content_loss = F.mse_loss(features[content_layer],target_content)
        style_loss = sum(F.mse_loss(gram(features[name]),target_style[name]) for name in style_layers)/len(style_layers)
        loss = content_loss + 1e6*style_loss
        assert torch.isfinite(loss)
        loss.backward(); optimizer.step()
        with torch.no_grad(): generated.clamp_(0,1)
        losses.append(loss.item())
    fig, axes = plt.subplots(1,3,figsize=(10,4))
    for ax,image,title in zip(axes,[content,style,generated],["Content","Procedural style","Optimized image"]):
        ax.imshow(image[0].detach().cpu().permute(1,2,0)); ax.set_title(title); ax.axis("off")
    fig.tight_layout(); fig.savefig(OUTPUT_DIR/"style_transfer.png",dpi=150,bbox_inches="tight"); plt.show()
    style_result = {"executed":True,"steps":len(losses),"first_loss":losses[0],"last_loss":losses[-1]}
    print(style_result)
else:
    print("Style transfer skipped. Set RUN_STYLE_TRANSFER=True to enable the optional experiment.")

## 8. Vision transformers: from images to patch tokens

Instead of convolutional feature maps, a vision transformer treats image patches as a sequence. The demonstration below uses a **tiny randomly initialized model** to show shapes, not useful predictions. It is not an ImageNet-pretrained ViT and has no measured classification quality.

A 32 × 32 RGB image with 8 × 8 patches gives 16 tokens, each initially containing 192 pixel values. A projection maps them to an embedding, and a class token plus positional embeddings are added before attention. Transfer-learning experiments with a pretrained ViT belong in a longer exercise.

In [ ]:
from torchvision.models.vision_transformer import VisionTransformer
small_image = F.interpolate(example_image.unsqueeze(0), size=(32,32), mode="bilinear", align_corners=False)
patches = F.unfold(small_image,kernel_size=8,stride=8).transpose(1,2)
tiny_vit = VisionTransformer(image_size=32,patch_size=8,num_layers=2,num_heads=4,
                             hidden_dim=64,mlp_dim=128,num_classes=2).eval()
with torch.no_grad():
    random_logits = tiny_vit(small_image)
assert patches.shape == (1,16,192) and random_logits.shape == (1,2)
print("Patch tensor:",tuple(patches.shape),"Output logits:",tuple(random_logits.shape))
print("Random weights: no meaningful prediction should be inferred.")

results = {"environment":versions,"data_url":DATA_URL,"archive_sha256":archive_sha256,
    "splits":[len(image_train),len(image_val),len(image_test)], "validation":validation_result,
    "test":test_result,"baseline_accuracy":baseline_accuracy,"best_epoch":best_epoch,
    "gradcam":cam_summaries,"checkpoint_roundtrip":True,"style_transfer":style_result,
    "vit_patch_shape":list(patches.shape),"elapsed_seconds":round(time.perf_counter()-run_started,2)}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(results,indent=2,allow_nan=False))
print("TUTORIAL_05_COMPLETE")
print(json.dumps(results,indent=2))

## 9. Exercises

1. Explain why `eval()` matters even when all backbone parameters have `requires_grad=False`.
2. Compare global average pooling with flattening the full feature map. How many classifier parameters would each need?
3. Generate Grad-CAM for another test image and both classes. Can a plausible heatmap accompany a wrong prediction?
4. Change the Grad-CAM target layer. What happens to spatial resolution and semantic detail?
5. In the optional style experiment, change the style weight by a factor of ten. Which visual features change? A lower objective is not necessarily a better-looking image.
6. As a separate validation-controlled experiment, unfreeze the final backbone block with a smaller learning rate. Recompute any cached features or train on images; stale cached features cannot represent a changing backbone.

**Concept checks:** Filters are parameters; activations depend on the input; Grad-CAM is class-specific; style transfer optimizes pixels; a randomly initialized transformer demonstrates mechanics but not recognition ability.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_05/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

Adapted and reworked for independent Colab use from [PEESE Tutorial 05](https://github.com/PEESEgroup/SysEn6888/tree/0b326ded95e415d7a60660879aaa737e3c7d79ea/Tutorial_05_CNN_in_Practice), Copyright (c) 2025 PEESE. The upstream MIT notice is retained in the repository `LICENSE`. This adaptation does not imply course endorsement.

Sources: [Google cats-and-dogs teaching dataset workflow](https://www.tensorflow.org/tutorials/images/transfer_learning), [MobileNetV2 weights and preprocessing](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.mobilenet_v2.html), [Grad-CAM paper](https://arxiv.org/abs/1610.02391), [PyTorch neural style tutorial](https://docs.pytorch.org/tutorials/advanced/neural_style_tutorial.html), and [torchvision VisionTransformer](https://docs.pytorch.org/vision/stable/models/vision_transformer.html). Images and pretrained weights are downloaded for runtime use and are not redistributed in this repository; their source terms apply.